# Laboratorium 8

**Imię i nazwisko:** Daniel Stefański  
**Nazwa ćwiczenia:** Analiza korpusu Wikipedii i statystyki leksykalne


In [ ]:
import re, math, warnings
warnings.filterwarnings('ignore')
from collections import Counter
from itertools import islice
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

wiki = load_dataset('wikimedia/wikipedia', '20231101.pl', split='train', streaming=True)
articles = list(islice(wiki, 200))
texts  = [a['text']  for a in articles]
titles = [a['title'] for a in articles]
print(f"Załadowano {len(texts)} artykułów.")

README.md: 0.00B [00:00, ?B/s]

Załadowano 200 artykułów.


### Zadanie 1: Pierwsza eksploracja korpusu
Korzystajac z zaladowanych 200 artykulow Wikipedii PL:

* Wyswietl tytuły pierwszych 10 artykulow.
* Dla kazdego z 10 artykulow wydrukuj: tytul, liczbe slow, pierwsze 2 zdania tekstu.
* Znajdz i wyswietl: najkrotszy i najdluzszy artykul (w slowach).
* W komórce Markdown: czym rozni sie korpus jezykowy od zbioru danych do
klasyfikacji?
(odpowiedz wlasnym slowami, 3–5 zdan)

In [ ]:
import re

print("Pierwsze 10 tytulow artykulow:")
for i, title in enumerate(titles[:10], start=1):
    print(f"{i}. {title}")

print()
print("Podsumowanie pierwszych 10 artykulow:")
for i, (title, text) in enumerate(zip(titles[:10], texts[:10]), start=1):
    words = re.findall(r"\S+", text)
    word_count = len(words)
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
    preview = " ".join(sentences[:2])

    print()
    print(f"[{i}] {title}")
    print(f"Liczba slow: {word_count}")
    print("Pierwsze 2 zdania:")
    print(preview)

all_word_counts = [len(re.findall(r"\S+", text)) for text in texts]
shortest_idx = min(range(len(all_word_counts)), key=lambda i: all_word_counts[i])
longest_idx = max(range(len(all_word_counts)), key=lambda i: all_word_counts[i])

print()
print("Najkrotszy artykul:")
print(f"Tytul: {titles[shortest_idx]}")
print(f"Liczba slow: {all_word_counts[shortest_idx]}")

print()
print("Najdluzszy artykul:")
print(f"Tytul: {titles[longest_idx]}")
print(f"Liczba slow: {all_word_counts[longest_idx]}")



Korpus jezykowy to duzy zbior tekstow zebranych po to, aby analizowac jezyk, jego slownictwo, struktury i czestotliwosci wystepowania roznych form. Taki korpus nie musi miec etykiet i zwykle sluzy bardziej do badan jezykowych albo uczenia modeli jezykowych. Zbior danych do klasyfikacji jest natomiast przygotowany pod konkretne zadanie i zawiera przyklady wraz z przypisanymi etykietami, na podstawie ktorych model uczy sie przewidywania klas. Krotko mowiac: korpus opisuje jezyk, a zbior do klasyfikacji pomaga rozwiazywac konkretne zadanie decyzyjne.


### Zadanie 2: Statystyki leksykalne i prawo Zipfa

Na pelnym zbiorze 200 artykulow:
* Polacz wszystkie teksty, przeprowadz prosta tokenizacje (split po bialych znakach,
lowercase, usun interpunkcje za pomoca re.sub).
* Oblicz i wyswietl: laczna liczba tokenow, liczba unikalnych tokenow (types), TTR.
* Wyswietl 20 najczestszych i 20 najrzadziej wystepujacych slow.
* Narysuj wykres prawa Zipfa: os X = ranga (log), os Y = czestotliwosc (log).

Na wykresie powinna wyjsc w przyblizeniu linia prosta — jesli nie, sprawdz tokenizacje.

In [ ]:
all_text = " ".join(texts).lower()
all_text = re.sub(r"[^\w\s-]", " ", all_text)
all_text = re.sub(r"_+", " ", all_text)
tokens = [tok for tok in all_text.split() if tok]

freq = Counter(tokens)
tokens_total = len(tokens)
types_total = len(freq)
ttr = types_total / tokens_total

print(f"Laczna liczba tokenow: {tokens_total}")
print(f"Liczba unikalnych tokenow (types): {types_total}")
print(f"TTR: {ttr:.4f}")

print()
print("20 najczestszych slow:")
for word, count in freq.most_common(20):
    print(f"{word:<20} {count}")

least_freq = min(freq.values())
rarest_words = sorted([word for word, count in freq.items() if count == least_freq])[:20]
print()
print("20 najrzadziej wystepujacych slow:")
for word in rarest_words:
    print(f"{word:<20} {freq[word]}")

ranks = np.arange(1, len(freq) + 1)
counts = np.array([count for _, count in freq.most_common()])

plt.figure(figsize=(8, 5))
plt.loglog(ranks, counts)
plt.title("Prawo Zipfa dla 200 artykulow Wikipedii PL")
plt.xlabel("Ranga (log)")
plt.ylabel("Czestotliwosc (log)")
plt.grid(True, which="both", alpha=0.3)
plt.show()


Prawo Zipfa oznacza praktycznie, ze bardzo mala grupa slow wystepuje ogromnie czesto, a bardzo duza liczba slow pojawia sie rzadko. Dla LLM ma to znaczenie, bo duza czesc budzetu tokenow zuzywa sie na powtarzajace sie, czeste elementy jezyka, a rzadkie slowa nadal musza byc obslugiwane przez slownik i kontekst modelu. Oznacza to, ze przy ograniczonym kontekscie trzeba madrze gospodarowac tokenami, bo dlugie teksty szybko zapelniaja okno kontekstowe. W praktyce pomaga to zrozumiec, dlaczego tokenizacja, kompresja informacji i selekcja tresci sa tak wazne.


### Zadanie 3: Rozklad dlugosci artykulow
* Oblicz dlugosc kazdego artykulu w slowach.
* Narysuj histogram (bins=30). Zaznacz mediame i P95 pionowymi liniami.
* Oblicz i wyswietl: min, max, srednia, mediana, odchylenie standardowe, P95.

In [ ]:
article_lengths = [len(re.findall(r"\S+", text)) for text in texts]
median_len = np.median(article_lengths)
p95_len = np.percentile(article_lengths, 95)

plt.figure(figsize=(8, 5))
plt.hist(article_lengths, bins=30, color="steelblue", edgecolor="black", alpha=0.8)
plt.axvline(median_len, color="orange", linestyle="--", linewidth=2, label=f"Mediana = {median_len:.1f}")
plt.axvline(p95_len, color="red", linestyle="--", linewidth=2, label=f"P95 = {p95_len:.1f}")
plt.title("Rozklad dlugosci artykulow")
plt.xlabel("Liczba slow")
plt.ylabel("Liczba artykulow")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Min: {np.min(article_lengths)}")
print(f"Max: {np.max(article_lengths)}")
print(f"Srednia: {np.mean(article_lengths):.2f}")
print(f"Mediana: {np.median(article_lengths):.2f}")
print(f"Odchylenie standardowe: {np.std(article_lengths):.2f}")
print(f"P95: {np.percentile(article_lengths, 95):.2f}")


Przy przyblizeniu 1 slowo ~ 1.3 tokenu limit 512 tokenow odpowiada mniej wiecej 394 slowom. To oznacza, ze obciecia wymagaloby okolo 111 z 200 artykulow. Jest to istotny problem, bo ponad polowa dokumentow nie zmiescilaby sie w calosci w pojedynczym wejsciu BERT-a. W praktyce oznacza to utrate czesci informacji albo koniecznosc dzielenia artykulow na fragmenty, co jest czestym rozwiazaniem przy dlugich tekstach.
